# Audiobook AI - Local Runner

Notebook này chạy local theo kiến trúc:

`Streamlit UI -> FastAPI API -> SQLite Queue -> XTTS Pipeline -> Download audio`

Chạy notebook từ repo local. FastAPI dùng `http://127.0.0.1:8000`, Streamlit dùng `http://127.0.0.1:8501`.

## 0. Chuẩn bị local

- Cài Python >= 3.10.
- Cài `ffmpeg` trên máy: `apt install ffmpeg`, `brew install ffmpeg`, hoặc `choco install ffmpeg`.
- Nếu dùng XTTS GPU, cần CUDA/Torch tương thích và GPU khả dụng.
- Nếu repo Hugging Face private/gated, đặt biến môi trường `HF_TOKEN` trước khi chạy notebook.

In [1]:
# [1] Xác định repo root
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'app.py').exists() and (candidate / 'streamlit_app.py').exists():
            return candidate
    raise RuntimeError('Không tìm thấy repo root chứa src/app.py và streamlit_app.py')

PROJECT_DIR = find_repo_root(Path.cwd())
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR =', PROJECT_DIR)
print('Python =', sys.version)


PROJECT_DIR = /workspace/CSC15012-Final-Project-Audiobook-Generation-System
Python = 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]


In [ ]:
# [2] Chọn nhanh TTS engine / model / thiết bị
# TTS_PRESET: 'aimy144_xtts', 'nguyenhoanganh_xtts', hoặc 'vieneu'
# TTS_DEVICE: 'auto', 'cuda', hoặc 'cpu'
import os
from pathlib import Path

TTS_PRESET = os.getenv('TTS_PRESET', 'aimy144_xtts').strip().lower()
TTS_DEVICE = os.getenv('TTS_DEVICE', 'auto').strip().lower()

if TTS_DEVICE not in {'auto', 'cuda', 'gpu', 'cpu'}:
    raise ValueError("TTS_DEVICE phải là 'auto', 'cuda'/'gpu', hoặc 'cpu'")

def detect_tts_device(requested: str) -> str:
    if requested == 'gpu':
        requested = 'cuda'
    if requested in {'cuda', 'cpu'}:
        return requested
    try:
        import torch
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    except Exception:
        return 'cpu'

EFFECTIVE_TTS_DEVICE = detect_tts_device(TTS_DEVICE)

XTTS_RUNTIME_REPO = 'https://github.com/nguyenhoanganh2002/XTTSv2-Finetuning-for-New-Languages.git'
XTTS_RUNTIME_DIR = PROJECT_DIR / 'models/XTTSv2-Finetuning-for-New-Languages'

TTS_PRESETS = {
    'aimy144_xtts': {
        'engine_family': 'xtts',
        'hf_repo': 'aiMy144/XTTSv2VietAudiobook',
        'local_dir': PROJECT_DIR / 'model',
    },
    'nguyenhoanganh_xtts': {
        'engine_family': 'xtts',
        'hf_repo': os.getenv('NGUYENHOANGANH_XTTS_HF_REPO', ''),
        'local_dir': PROJECT_DIR / 'scripts/model',
    },
    'vieneu': {
        'engine_family': 'vieneu',
        'hf_repo': '',
        'local_dir': None,
    },
}

if TTS_PRESET not in TTS_PRESETS:
    raise ValueError(f'TTS_PRESET không hợp lệ: {TTS_PRESET}. Chọn một trong: {list(TTS_PRESETS)}')

selected = TTS_PRESETS[TTS_PRESET]
if selected['engine_family'] == 'xtts':
    TTS_ENGINE = 'xtts_gpu' if EFFECTIVE_TTS_DEVICE == 'cuda' else 'xtts_cpu'
    XTTS_HF_REPO = os.getenv('XTTS_HF_REPO') or selected['hf_repo']
    XTTS_LOCAL_DIR = Path(os.getenv('XTTS_MODEL_NAME_OR_PATH') or selected['local_dir']).resolve()
else:
    TTS_ENGINE = 'vieneu'
    XTTS_HF_REPO = os.getenv('XTTS_HF_REPO', 'aiMy144/XTTSv2VietAudiobook')
    XTTS_LOCAL_DIR = Path(os.getenv('XTTS_MODEL_NAME_OR_PATH') or (PROJECT_DIR / 'model')).resolve()

VIENEU_MODEL_NAME = os.getenv('VIENEU_MODEL_NAME', 'pnnbao-ump/VieNeu-TTS-v2')
VIENEU_MODE = os.getenv('VIENEU_MODE', 'standard')
VIENEU_EMOTION = os.getenv('VIENEU_EMOTION', 'storytelling')
VIENEU_DEVICE = os.getenv('VIENEU_DEVICE', EFFECTIVE_TTS_DEVICE)

os.environ.update({
    'TTS_PRESET': TTS_PRESET,
    'TTS_ENGINE': TTS_ENGINE,
    'TTS_DEVICE': EFFECTIVE_TTS_DEVICE,
    'XTTS_DEVICE': EFFECTIVE_TTS_DEVICE,
    'XTTS_HF_REPO': XTTS_HF_REPO,
    'XTTS_MODEL_NAME_OR_PATH': str(XTTS_LOCAL_DIR),
    'XTTS_RUNTIME_DIR': str(XTTS_RUNTIME_DIR),
    'VIENEU_MODEL_NAME': VIENEU_MODEL_NAME,
    'VIENEU_MODE': VIENEU_MODE,
    'VIENEU_EMOTION': VIENEU_EMOTION,
    'VIENEU_DEVICE': VIENEU_DEVICE,
})

print('TTS preset =', TTS_PRESET)
print('TTS engine =', TTS_ENGINE)
print('TTS device =', EFFECTIVE_TTS_DEVICE)
print('XTTS HF repo =', XTTS_HF_REPO or '(local only)')
print('XTTS model dir =', XTTS_LOCAL_DIR)
print('VieNeu model =', VIENEU_MODEL_NAME)


In [2]:
# [2] Kiểm tra GPU và ffmpeg
import shutil
import subprocess
import sys

print('ffmpeg =', shutil.which('ffmpeg') or 'NOT FOUND')
if not shutil.which('ffmpeg'):
    print('Cần cài ffmpeg trước khi xuất mp3 hoặc xử lý audio.')

try:
    import torch
    print('Torch =', torch.__version__)
    print('CUDA available =', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU =', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


ffmpeg = /usr/bin/ffmpeg


Torch = 2.12.0+cu130
CUDA available = True
GPU = NVIDIA GeForce RTX 4060 Ti


In [3]:
pip install ffmpeg-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.3/491.3 KB 8.7 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
# [4] Cài/cập nhật dependencies local
# Nếu bạn đang dùng virtualenv/conda, hãy chọn đúng kernel trước khi chạy cell này.
import subprocess
import sys
import os
import shutil
from pathlib import Path

def ensure_vieneu_build_tools():
    missing = [cmd for cmd in ('cc', 'gcc', 'g++', 'cmake') if shutil.which(cmd) is None]
    if shutil.which('espeak-ng') is None:
        missing.append('espeak-ng')
    if not missing:
        return
    if os.name != 'posix' or shutil.which('apt-get') is None:
        raise RuntimeError(
            'Thiếu build tools cho vieneu: ' + ', '.join(missing) +
            '. Hãy cài C/C++ compiler, cmake và espeak-ng trước khi pip install.'
        )
    apt = shutil.which('apt-get')
    sudo = shutil.which('sudo')
    prefix = [sudo] if sudo and os.geteuid() != 0 else []
    subprocess.run(prefix + [apt, 'update'], check=True)
    subprocess.run(prefix + [apt, 'install', '-y', 'build-essential', 'gcc', 'g++', 'cmake', 'pkg-config', 'ninja-build', 'espeak-ng'], check=True)

TTS_ENGINE_SELECTED = os.getenv('TTS_ENGINE', globals().get('TTS_ENGINE', 'xtts_gpu')).lower()
TTS_DEVICE_SELECTED = os.getenv('TTS_DEVICE', globals().get('EFFECTIVE_TTS_DEVICE', 'auto')).lower()
if TTS_DEVICE_SELECTED == 'auto':
    try:
        import torch
        TTS_DEVICE_SELECTED = 'cuda' if torch.cuda.is_available() else 'cpu'
    except Exception:
        TTS_DEVICE_SELECTED = 'cpu'
XTTS_ENGINES = {'xtts', 'xtts_gpu', 'xtts_cpu', 'direct_xtts'}

if TTS_ENGINE_SELECTED in {'vieneu', 'vieneu_tts', 'direct_vieneu'}:
    ensure_vieneu_build_tools()

XTTS_RUNTIME_REPO = globals().get('XTTS_RUNTIME_REPO', 'https://github.com/nguyenhoanganh2002/XTTSv2-Finetuning-for-New-Languages.git')
XTTS_RUNTIME_DIR = Path(os.getenv('XTTS_RUNTIME_DIR', globals().get('XTTS_RUNTIME_DIR', Path('models/XTTSv2-Finetuning-for-New-Languages'))))

if TTS_ENGINE_SELECTED in XTTS_ENGINES and not XTTS_RUNTIME_DIR.exists():
    subprocess.run(['git', 'clone', XTTS_RUNTIME_REPO, str(XTTS_RUNTIME_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
if TTS_ENGINE_SELECTED in XTTS_ENGINES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(XTTS_RUNTIME_DIR / 'requirements.txt')], check=True)
elif TTS_ENGINE_SELECTED in {'vieneu', 'vieneu_tts', 'direct_vieneu'}:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vieneu'], check=True)

# TorchCodec is ABI-sensitive: torchcodec 0.11.x is built for torch 2.11.x.
# Use CUDA wheels for torch/torchaudio, but CPU-only torchcodec to avoid requiring
# CUDA Toolkit libraries such as libnppicc.so.13 in local venv/docker runtimes.
# Reinstall the PyTorch stack after XTTS requirements so transitive installs cannot
# leave torch/torchaudio/torchcodec on mismatched versions.
if TTS_DEVICE_SELECTED == 'cpu':
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        '--index-url', 'https://download.pytorch.org/whl/cpu',
        'torch==2.11.0', 'torchaudio==2.11.0',
    ], check=True)
else:
    PYTORCH_CUDA_INDEX = 'https://download.pytorch.org/whl/cu130'
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        '--index-url', PYTORCH_CUDA_INDEX,
        'torch==2.11.0+cu130', 'torchaudio==2.11.0+cu130',
    ], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
    '--index-url', 'https://download.pytorch.org/whl/cpu',
    'torchcodec==0.11.1',
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-multipart', 'huggingface_hub', 'hf_transfer'], check=True)

import fastapi, streamlit
print('Dependencies OK')


Cloning into 'models/XTTSv2-Finetuning-for-New-Languages'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.57.0 requires numpy<3,>=1.23, but you have numpy 1.22.0 which is incompatible.


Dependencies OK


In [3]:
# [5] Tải model TTS và tạo .env local
from pathlib import Path
import os
import sys

from huggingface_hub import snapshot_download


def has_xtts_model_files(model_dir: Path) -> bool:
    return (
        model_dir.exists()
        and (model_dir / 'config.json').exists()
        and (model_dir / 'vocab.json').exists()
        and any(model_dir.glob('*.pth'))
    )

TTS_PRESET = os.getenv('TTS_PRESET', globals().get('TTS_PRESET', 'aimy144_xtts'))
TTS_ENGINE = os.getenv('TTS_ENGINE', globals().get('TTS_ENGINE', 'xtts_gpu'))
TTS_DEVICE = os.getenv('TTS_DEVICE', globals().get('EFFECTIVE_TTS_DEVICE', 'auto'))
XTTS_DEVICE = os.getenv('XTTS_DEVICE', TTS_DEVICE)
VIENEU_MODEL_NAME = os.getenv('VIENEU_MODEL_NAME', globals().get('VIENEU_MODEL_NAME', 'pnnbao-ump/VieNeu-TTS-v2'))
VIENEU_MODE = os.getenv('VIENEU_MODE', globals().get('VIENEU_MODE', 'standard'))
VIENEU_EMOTION = os.getenv('VIENEU_EMOTION', globals().get('VIENEU_EMOTION', 'storytelling'))
VIENEU_DEVICE = os.getenv('VIENEU_DEVICE', TTS_DEVICE)
XTTS_HF_REPO = os.getenv('XTTS_HF_REPO', globals().get('XTTS_HF_REPO', 'aiMy144/XTTSv2VietAudiobook'))
DEFAULT_MODEL_DIR = PROJECT_DIR / 'model'
SCRIPTS_MODEL_DIR = PROJECT_DIR / 'scripts/model'
XTTS_LOCAL_DIR = Path(os.getenv('XTTS_MODEL_NAME_OR_PATH') or '') if os.getenv('XTTS_MODEL_NAME_OR_PATH') else None

if XTTS_LOCAL_DIR is None:
    XTTS_LOCAL_DIR = SCRIPTS_MODEL_DIR if has_xtts_model_files(SCRIPTS_MODEL_DIR) else DEFAULT_MODEL_DIR
XTTS_LOCAL_DIR = XTTS_LOCAL_DIR.resolve()

XTTS_RUNTIME_DIR = PROJECT_DIR / 'models/XTTSv2-Finetuning-for-New-Languages'
HF_TOKEN = os.getenv('HF_TOKEN') or None

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

XTTS_ENGINES = {'xtts', 'xtts_gpu', 'xtts_cpu', 'direct_xtts'}

if TTS_ENGINE in XTTS_ENGINES and not has_xtts_model_files(XTTS_LOCAL_DIR):
    if not XTTS_HF_REPO:
        raise RuntimeError(
            f'Không tìm thấy model XTTS local ở {XTTS_LOCAL_DIR}. '
            'Với preset nguyenhoanganh_xtts, hãy đặt model vào scripts/model '
            'hoặc set NGUYENHOANGANH_XTTS_HF_REPO / XTTS_HF_REPO.'
        )
    snapshot_download(
        repo_id=XTTS_HF_REPO,
        repo_type='model',
        local_dir=str(XTTS_LOCAL_DIR),
        token=HF_TOKEN,
        max_workers=8,
    )

if TTS_ENGINE in XTTS_ENGINES:
    if str(XTTS_RUNTIME_DIR) not in sys.path:
        sys.path.insert(0, str(XTTS_RUNTIME_DIR))
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts

STORAGE_DIR = PROJECT_DIR / 'storage'
VOICE_DIR = PROJECT_DIR / 'data/voice_samples'
STORAGE_DIR.mkdir(exist_ok=True)
VOICE_DIR.mkdir(parents=True, exist_ok=True)

env_text = f'''API_BASE_URL=http://127.0.0.1:8000
CORS_ORIGINS=http://localhost:8501,http://127.0.0.1:8501
STORAGE_DIR={STORAGE_DIR}
MAX_UPLOAD_MB=200
MAX_CONCURRENT_JOBS=1
TTS_PRESET={TTS_PRESET}
TTS_ENGINE={TTS_ENGINE}
TTS_DEVICE={TTS_DEVICE}
XTTS_DEVICE={XTTS_DEVICE}
XTTS_HF_REPO={XTTS_HF_REPO}
VIENEU_MODEL_NAME={VIENEU_MODEL_NAME}
VIENEU_MODE={VIENEU_MODE}
VIENEU_EMOTION={VIENEU_EMOTION}
VIENEU_DEVICE={VIENEU_DEVICE}
VIENEU_API_BASE={os.getenv('VIENEU_API_BASE', '')}
XTTS_MODEL_NAME_OR_PATH={XTTS_LOCAL_DIR}
XTTS_CONFIG_PATH=
XTTS_VOCAB_PATH=
XTTS_RUNTIME_DIR={XTTS_RUNTIME_DIR}
XTTS_VOICE_DIR={VOICE_DIR}
HF_TOKEN={HF_TOKEN or ''}
'''
(PROJECT_DIR / '.env').write_text(env_text, encoding='utf-8')
print('.env created at', PROJECT_DIR / '.env')
print('TTS preset =', TTS_PRESET)
print('TTS engine =', TTS_ENGINE)
print('TTS device =', TTS_DEVICE)
print('VieNeu model =', VIENEU_MODEL_NAME)
print('XTTS HF repo =', XTTS_HF_REPO or '(local only)')
print('XTTS model dir =', XTTS_LOCAL_DIR)
print('Voice dir =', VOICE_DIR)


/workspace/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


.env created at /workspace/CSC15012-Final-Project-Audiobook-Generation-System/.env
XTTS model dir = /workspace/CSC15012-Final-Project-Audiobook-Generation-System/model
Voice dir = /workspace/CSC15012-Final-Project-Audiobook-Generation-System/data/voice_samples


## 5. Reference Voices

XTTS cần ít nhất một file `.wav` trong `data/voice_samples`. Repo có sẵn vài file mẫu; có thể thêm file WAV khác vào folder này.

In [13]:
# [5] Kiểm tra reference voices
from pathlib import Path

voice_dir = PROJECT_DIR / 'data/voice_samples'
voices = sorted(p.name for p in voice_dir.glob('*.wav'))
print(f'Found {len(voices)} reference voices in {voice_dir}')
for name in voices:
    print('-', name)

assert voices, f'Thiếu reference voice WAV trong {voice_dir}'


Found 6 reference voices in /workspace/CSC15012-Final-Project-Audiobook-Generation-System/data/voice_samples
- female_hcm_01.wav
- female_hcm_02.wav
- female_hn_01.wav
- female_hn_02.wav
- male_hn_01.wav
- male_hn_02.wav


In [14]:
# [6] Chạy FastAPI local
import os
import signal
import socket
import subprocess
import sys
import threading
import time
from pathlib import Path

API_PORT = 8000
API_BASE_URL = f'http://127.0.0.1:{API_PORT}'
api_log = []
api_processes = globals().setdefault('api_processes', [])


def stop_processes(processes):
    for proc in list(processes):
        if proc.poll() is not None:
            continue
        try:
            if os.name == 'nt':
                proc.terminate()
            else:
                os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        except Exception:
            proc.terminate()
    for proc in list(processes):
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
    processes.clear()


def ready(port):
    try:
        socket.create_connection(('127.0.0.1', port), timeout=1).close()
        return True
    except OSError:
        return False


def resolve_xtts_model_dir() -> Path:
    env_model = os.getenv('XTTS_MODEL_NAME_OR_PATH')
    if env_model:
        return Path(env_model).resolve()
    if 'XTTS_LOCAL_DIR' in globals():
        return Path(XTTS_LOCAL_DIR).resolve()
    scripts_model = PROJECT_DIR / 'scripts/model'
    if scripts_model.exists() and any(scripts_model.glob('*.pth')):
        return scripts_model.resolve()
    return (PROJECT_DIR / 'model').resolve()

stop_processes(api_processes)

def start_api():
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_DIR) + os.pathsep + env.get('PYTHONPATH', '')
    env['API_BASE_URL'] = API_BASE_URL
    env['TTS_ENGINE'] = os.getenv('TTS_ENGINE', 'xtts_gpu')
    env['TTS_DEVICE'] = os.getenv('TTS_DEVICE', 'auto')
    env['XTTS_DEVICE'] = os.getenv('XTTS_DEVICE', env['TTS_DEVICE'])
    env['XTTS_HF_REPO'] = os.getenv('XTTS_HF_REPO', '')
    env['VIENEU_MODEL_NAME'] = os.getenv('VIENEU_MODEL_NAME', 'pnnbao-ump/VieNeu-TTS-v2')
    env['VIENEU_MODE'] = os.getenv('VIENEU_MODE', 'standard')
    env['VIENEU_EMOTION'] = os.getenv('VIENEU_EMOTION', 'storytelling')
    env['VIENEU_DEVICE'] = os.getenv('VIENEU_DEVICE', env['TTS_DEVICE'])
    env['VIENEU_API_BASE'] = os.getenv('VIENEU_API_BASE', '')
    env['STORAGE_DIR'] = str(PROJECT_DIR / 'storage')
    env['XTTS_MODEL_NAME_OR_PATH'] = str(resolve_xtts_model_dir())
    env['XTTS_RUNTIME_DIR'] = str(PROJECT_DIR / 'models/XTTSv2-Finetuning-for-New-Languages')
    env['XTTS_VOICE_DIR'] = str(PROJECT_DIR / 'data/voice_samples')

    creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == 'nt' else 0
    p = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'src.app:app', '--host', '127.0.0.1', '--port', str(API_PORT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        cwd=PROJECT_DIR,
        start_new_session=(os.name != 'nt'),
        creationflags=creationflags,
    )
    api_processes.append(p)
    for line in p.stdout:
        api_log.append(line.rstrip())

threading.Thread(target=start_api, daemon=True).start()

print('XTTS model dir =', resolve_xtts_model_dir())
print('Waiting for FastAPI', end='')
for _ in range(120):
    time.sleep(1)
    print('.', end='', flush=True)
    if ready(API_PORT):
        break
print()

assert ready(API_PORT), 'FastAPI chưa lên. Chạy cell logs bên dưới để xem lỗi.'
print('FastAPI local OK:', API_BASE_URL)


XTTS model dir = /workspace/CSC15012-Final-Project-Audiobook-Generation-System/model
Waiting for FastAPI

..
FastAPI local OK: http://127.0.0.1:8000


In [15]:
# [7] Chạy Streamlit local
import os
import signal
import socket
import subprocess
import sys
import threading
import time

UI_PORT = 8501
UI_URL = f'http://127.0.0.1:{UI_PORT}'
ui_log = []
ui_processes = globals().setdefault('ui_processes', [])

stop_processes(ui_processes)

def start_streamlit():
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_DIR) + os.pathsep + env.get('PYTHONPATH', '')
    env['API_BASE_URL'] = API_BASE_URL
    env['STREAMLIT_BROWSER_GATHER_USAGE_STATS'] = 'false'
    env['STREAMLIT_GLOBAL_SHOW_WARNING_ON_DIRECT_EXECUTION'] = 'false'

    creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == 'nt' else 0
    p = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'streamlit_app.py',
         f'--server.port={UI_PORT}', '--server.headless=true',
         '--server.enableCORS=false', '--server.enableXsrfProtection=false',
         '--browser.gatherUsageStats=false'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        cwd=PROJECT_DIR,
        start_new_session=(os.name != 'nt'),
        creationflags=creationflags,
    )
    ui_processes.append(p)
    for line in p.stdout:
        ui_log.append(line.rstrip())

threading.Thread(target=start_streamlit, daemon=True).start()

print('Waiting for Streamlit', end='')
for _ in range(60):
    time.sleep(1)
    print('.', end='', flush=True)
    if ready(UI_PORT):
        break
print()

assert ready(UI_PORT), 'Streamlit chưa lên. Chạy cell logs bên dưới để xem lỗi.'
print('=' * 80)
print('OPEN STREAMLIT:', UI_URL)
print('FastAPI:', API_BASE_URL)
print('=' * 80)


Waiting for Streamlit.
OPEN STREAMLIT: http://127.0.0.1:8501
FastAPI: http://127.0.0.1:8000


In [17]:
# [8] Xem logs khi cần debug
print('=== API LOG ===')
print('\n'.join(api_log[-200:]) or '(empty)')
print('\n=== STREAMLIT LOG ===')
print('\n'.join(ui_log[-200:]) or '(empty)')


=== API LOG ===
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.12.0+cu130) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:

[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call last):
  File "/workspace/venv/lib/python3.10/site-packages/torch/_ops.py", line 1509, in load_library
    ctypes.CDLL(path)
  

In [12]:
# [9] Dừng FastAPI / Streamlit khi cần
for processes in (globals().get('api_processes', []), globals().get('ui_processes', [])):
    if 'stop_processes' in globals():
        stop_processes(processes)

print('Stopped FastAPI and Streamlit.')


Stopped FastAPI and Streamlit.


## Cách test local

1. Chạy lần lượt các cell `[1]` đến `[7]`.
2. Mở `http://127.0.0.1:8501` trong trình duyệt.
3. Upload EPUB, chọn định dạng, rồi bấm `Create audiobook job`.
4. Nếu lỗi, chạy cell `[8]` để xem API và Streamlit logs.